# Brent · `appeal_decision_date` (web: "Appeal decision date")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and values that are not dates

Fetch `appeal_decision_date` for every Brent application (paged with `search_after`,
sorted by `id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it;
- **not a date**: a value that doesn't parse as `DD/MM/YYYY`.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "appeal_decision_date"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "appeal_decision_date"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["appeal_decision_date"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")
parsed = pd.to_datetime(value, format="%d/%m/%Y", errors="coerce")
not_a_date = value.notna() & ~is_blank & parsed.isna()

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "not a date": int(not_a_date.sum()),
    "valid DD/MM/YYYY date": int(parsed.notna().sum()),
})
summary["total"] = summary.sum()
display(summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)}))

print("Values that are not dates:")
df[not_a_date]

,count,share_%
null,35648,96.47
empty or blank,0,0.00
not a date,0,0.00
valid DD/MM/YYYY date,1304,3.53
total,36952,100.00


Values that are not dates:


,id,appeal_decision_date


### Findings · Step 1

Live API, 2026-11-10, 36,952 Brent applications:

| | Count | Share |
|---|---|---|
| null | 35,648 | 96.47% |
| empty or blank | 0 | 0% |
| not a date | 0 | 0% |
| valid `DD/MM/YYYY` date | 1,304 | 3.53% |

- **Clean format:** every filled value is a real `DD/MM/YYYY` date.
- 1,304 have a date; 1 record has an `appeal_decision` but no date (see the
  `appeal_decision` notebook, Step 1).

## Step 2 · Do the values match the schema type?

The schema gives `appeal_decision_date` the type `date` (`DD/MM/YYYY`). JSON has no date
type, so a date should come as a string; Step 1 already showed every filled value parses.
Count the Python type of every value in `df`: `str` = JSON string, `int` / `float` = JSON
number, `NoneType` = null.

In [4]:
types = df["appeal_decision_date"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
display(types.value_counts().to_frame("count"))

not_str = df[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))
not_str["appeal_decision_date"].value_counts().head(30).to_frame("count")

,count
appeal_decision_date,
NoneType,35648
str,1304


values that are not strings: 0


,count
appeal_decision_date,


### Findings · Step 2

Live API, 2026-11-10:

| Python type | Count |
|---|---|
| `str` | 1,304 |
| null | 35,648 |
| number | 0 |

- **Every filled value is a JSON string** in `DD/MM/YYYY` (Step 1), which is how the
  schema's `date` type comes in JSON. No type mismatch.

## Step 3 · Most common values

Count identical dates in `df` and show the 20 most common, plus the earliest and latest
date.

In [5]:
dates = pd.to_datetime(df["appeal_decision_date"], format="%d/%m/%Y", errors="coerce")
print("distinct dates:", dates.nunique(), "| earliest:", dates.min().date(), "| latest:", dates.max().date())
display(dates.dt.year.value_counts().sort_index().to_frame("appeals per year"))
df["appeal_decision_date"].value_counts().head(20).to_frame("count")

distinct dates: 903 | earliest: 1987-03-01 | latest: 2026-11-03


,appeals per year
appeal_decision_date,
1987.0,6
1988.0,1
1989.0,4
1990.0,1
1991.0,4
1994.0,2
1996.0,2
1998.0,1
1999.0,2


,count
appeal_decision_date,
12/09/2023,8
08/10/2020,7
19/05/2021,7
29/04/2022,7
29/11/2022,7
22/09/2022,6
30/03/2022,6
25/06/2025,6
06/11/2019,5


### Findings · Step 3

Live API, 2026-11-10: 1,304 dates, 903 distinct, from 01/03/1987 to 03/11/2026.

- **No placeholder dates:** the most common date is used only 8 times (12/09/2023).
- **Mostly recent:** about 120-180 appeal decisions a year from 2019; before 2018 only
  0-18 a year. This fits Datahub coverage starting around 2018.
- The earliest, 01/03/1987, is one of the 2 same-day legacy appeals in `decision_date`
  (open question 8).